In [39]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans

In [40]:
INPUT_CSV   = "../data/modular_mapping_approach/oganilir_modular_td/Epistem_element_props_extracted_table.csv"   # exported from GEE
OUTPUT_PROFILES  = "../data/modular_mapping_approach/oganilir_modular_td/cluster_profiles_oganilir.csv"
OUTPUT_CLUSTERED = "../data/modular_mapping_approach/oganilir_modular_td/td_clustered.csv"
 
PRIMITIVE_COLUMNS = [
    # "treePresenceType",
    "treecover",
    "treeheight",
    "lai_mean",
    # "treewoodyLeafPhenology",
    # "woodyLeafType",
    # "shrubPresenceType",
    # "herbaceousPresenceType",
    # "builtUpPresenceType",
    # "naturalSurface_presence",
    # "bareSoilPresence",
    # "water_presence",
    # "waterSeasonality",
    # "vegetationArtificiality",
    # "crop_cycles",
]
 
K    = 13   # number of clusters — adjust to ~n_classes × 2
SEED = 42  # keep identical to Script B for reproducibility

In [41]:
td = pd.read_csv(INPUT_CSV)
td = pd.read_csv(INPUT_CSV)
print(td.columns.tolist())
print(td.dtypes)


['ID_epistem', 'name_epist', 'treePresenceType', 'treecover', 'treeheight', 'lai_mean', 'treewoodyLeafPhenology', 'woodyLeafType', 'shrubPresenceType', 'herbaceousPresenceType', 'builtUpPresenceType', 'built_nres_m2', 'naturalSurface_presence', 'bareSoilPresence', 'water_presence', 'waterSeasonality', 'vegetationArtificiality', 'crop_intensity', 'crop_cycles']
ID_epistem                   int64
name_epist                  object
treePresenceType           float64
treecover                  float64
treeheight                 float64
lai_mean                   float64
treewoodyLeafPhenology     float64
woodyLeafType              float64
shrubPresenceType          float64
herbaceousPresenceType     float64
builtUpPresenceType        float64
built_nres_m2              float64
naturalSurface_presence    float64
bareSoilPresence           float64
water_presence             float64
waterSeasonality           float64
vegetationArtificiality    float64
crop_intensity             float64
crop_cy

In [42]:

def check_value_type(series, threshold=3):
    if pd.api.types.is_numeric_dtype(series):
        unique_vals = series.nunique(dropna=True)
        
        # integers with few unique values = likely discrete
        if pd.api.types.is_integer_dtype(series) and unique_vals < threshold:
            return "discrete"
        
        # floats with many unique values = likely continuous
        if unique_vals >= threshold:
            return "continuous"
        else:
            return "discrete"
    
    else:
        return "categorical (discrete)"

for col in td.columns:
    col_type = td[col].dtype
    nature = check_value_type(td[col])
    print(f"{col}: {col_type} -> {nature}")

ID_epistem: int64 -> continuous
name_epist: object -> categorical (discrete)
treePresenceType: float64 -> discrete
treecover: float64 -> continuous
treeheight: float64 -> continuous
lai_mean: float64 -> continuous
treewoodyLeafPhenology: float64 -> continuous
woodyLeafType: float64 -> discrete
shrubPresenceType: float64 -> discrete
herbaceousPresenceType: float64 -> discrete
builtUpPresenceType: float64 -> discrete
built_nres_m2: float64 -> discrete
naturalSurface_presence: float64 -> discrete
bareSoilPresence: float64 -> discrete
water_presence: float64 -> continuous
waterSeasonality: float64 -> continuous
vegetationArtificiality: float64 -> discrete
crop_intensity: float64 -> continuous
crop_cycles: float64 -> continuous


In [43]:

null_counts_before = td[PRIMITIVE_COLUMNS].isnull().sum()
print("\nNull counts before imputation:")
print(null_counts_before[null_counts_before > 0].to_string() or "  none")
 
td[PRIMITIVE_COLUMNS] = td[PRIMITIVE_COLUMNS].fillna(0)
 
assert td[PRIMITIVE_COLUMNS].isnull().sum().sum() == 0, \
    "Nulls remain after imputation — check input data"
print("Null counts after imputation: all zero ✓")


Null counts before imputation:
treecover    34
lai_mean     21
Null counts after imputation: all zero ✓


In [44]:
scaler = MinMaxScaler()
norm_columns = [c + "_norm" for c in PRIMITIVE_COLUMNS]
 
td[norm_columns] = scaler.fit_transform(td[PRIMITIVE_COLUMNS])
 
print("\nNormalised range check (all should be [0.0, 1.0]):")
print(td[norm_columns].agg(["min", "max"]).T.to_string())


Normalised range check (all should be [0.0, 1.0]):
                 min  max
treecover_norm   0.0  1.0
treeheight_norm  0.0  1.0
lai_mean_norm    0.0  1.0


In [45]:
kmeans = KMeans(
    n_clusters=K,
    random_state=SEED,
    n_init=10,
    max_iter=300,
)
td["cluster_id"] = kmeans.fit_predict(td[norm_columns])
 
print(f"\nCluster distribution (k={K}):")
print(td["cluster_id"].value_counts().sort_index().to_string())
 
# # Save full clustered TD — used in Script B to re-attach labels
# td.to_csv(OUTPUT_CLUSTERED, index=False)
# print(f"\nFull clustered dataset saved → {OUTPUT_CLUSTERED}")


Cluster distribution (k=13):
cluster_id
0      18
1      38
2     168
3      43
4      40
5      33
6      32
7       2
8      57
9      57
10     55
11    133
12     24


c:\Users\widijanto\AppData\Local\miniconda3\envs\phdprops\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=3.
  warnings.warn(


In [46]:
profile_rows = []
 
for cluster_id in range(K):
    subset = td[td["cluster_id"] == cluster_id][PRIMITIVE_COLUMNS]
    row = {"cluster_id": cluster_id, "n_samples": len(subset)}
 
    for col in PRIMITIVE_COLUMNS:
        row[f"{col}_p25"] = subset[col].quantile(0.25)
        row[f"{col}_p50"] = subset[col].quantile(0.50)  # median
        row[f"{col}_p75"] = subset[col].quantile(0.75)
 
    # Blank columns for manual semantic matching
    row["class_id"] = ""
    row["notes"]    = ""
 
    profile_rows.append(row)
 
profiles = pd.DataFrame(profile_rows)

stat_cols = []
for col in PRIMITIVE_COLUMNS:
    stat_cols += [f"{col}_p50", f"{col}_p25", f"{col}_p75"]
 
col_order = ["cluster_id", "n_samples"] + stat_cols + ["class_id", "notes"]
profiles = profiles[col_order]
 
print("\nCluster profiles preview:")
print(profiles[["cluster_id", "n_samples", "class_id"]].to_string(index=False))
 
# profiles.to_csv(OUTPUT_PROFILES, index=False)


Cluster profiles preview:
 cluster_id  n_samples class_id
          0         18         
          1         38         
          2        168         
          3         43         
          4         40         
          5         33         
          6         32         
          7          2         
          8         57         
          9         57         
         10         55         
         11        133         
         12         24         


In [47]:
import plotly.express as px

melted = td.melt(
    id_vars=["cluster_id"],
    value_vars=PRIMITIVE_COLUMNS,
    var_name="feature",
    value_name="value"
)

fig = px.box(
    melted,
    x="feature",
    y="value",
    color="cluster_id",
    title="Feature Distributions per Cluster",
    points="outliers"
)

fig.update_layout(
    template="plotly_white",
    xaxis_tickangle=-45
)

fig.show()

In [56]:
# --- ensure columns exist ---
required_cols = ["cluster_id", "ID_epistem", "name_epist"]
missing = [c for c in required_cols if c not in td.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

# --- 1. cluster to epistem distribution ---
cluster_epistem_counts = (
    td.groupby(["cluster_id", "ID_epistem"])
    .size()
    .reset_index(name="count")
)

# --- 2. find dominant ID_epistem per cluster ---
dominant_mapping = (
    cluster_epistem_counts
    .sort_values(["cluster_id", "count"], ascending=[True, False])
    .drop_duplicates("cluster_id")
    .rename(columns={
        "ID_epistem": "dominant_ID_epistem",
        "count": "dominant_count"
    })
)

# --- 3. compute which one of the clusters is most dominant for each ID_epistem ---
cluster_totals = td.groupby("cluster_id").size().reset_index(name="total")
dominant_mapping = dominant_mapping.merge(cluster_totals, on="cluster_id")
dominant_mapping["dominance"] = dominant_mapping["dominant_count"] / dominant_mapping["total"]

# --- 4. attach name_epist ---
name_lookup = td[["ID_epistem", "name_epist"]].drop_duplicates()

dominant_mapping = dominant_mapping.merge(
    name_lookup,
    left_on="dominant_ID_epistem",
    right_on="ID_epistem",
    how="left"
).drop(columns=["ID_epistem"])

# --- 6. summary stats ---
print("\n===== OVERALL SUMMARY =====")
print(f"Mean cluster dominance: {dominant_mapping['dominance'].mean():.3f}")
print(f"Min dominance: {dominant_mapping['dominance'].min():.3f}")
print(f"Max dominance: {dominant_mapping['dominance'].max():.3f}")

# check missing classes
all_epistem = set(td["ID_epistem"].unique())
mapped_epistem = set(dominant_mapping["dominant_ID_epistem"].dropna())

missing_epistem = all_epistem - mapped_epistem

# recommendation rules ---
def assign_label(row):
    if row["dominance"] >= 0.80:
        return "HIGH_CONFIDENCE"
    elif row["dominance"] >= 0.60:
        return "MEDIUM_CONFIDENCE"
    elif row["dominance"] >= 0.40:
        return "LOW_CONFIDENCE"
    else:
        return "NO CLEAR CLASS"

dominant_mapping["confidence_label"] = dominant_mapping.apply(assign_label, axis=1)

print("\n===== LABEL RECOMMENDATION =====")

print(
    dominant_mapping.sort_values("cluster_id", ascending=True)[
        [
            "cluster_id",
            "name_epist",
            # "dominant_ID_epistem",
            "dominant_count",
            "total",
            "dominance",
            "confidence_label"
        ]
    ].to_string(index=False)
)

print("\n===== CLASSES NOT CAPTURED BY ANY CLUSTER =====")
# hard coded: check the missing classes where they are second or third best in any cluster, and if so, suggest assignment to that cluster
if len(missing_epistem) == 0:
    print("None — all epistem classes appear in at least one cluster ✓")
else:
    # For each missing class, find its top-3 clusters by count
    missing_suggestions = []
    
    for epistem_id in sorted(missing_epistem):
        epistem_name = td[td["ID_epistem"] == epistem_id]["name_epist"].iloc[0]
        
        # Get cluster counts for this epistemic class
        class_in_clusters = (
            td[td["ID_epistem"] == epistem_id]
            .groupby("cluster_id")
            .size()
            .reset_index(name="count")
            .sort_values("count", ascending=False)
        )
        
        # Get top 3 clusters
        top_clusters = class_in_clusters.head(3)
        
        # For each top cluster, get the dominant class and dominance ratio
        for idx, row in top_clusters.iterrows():
            cluster_id = row["cluster_id"]
            count_in_cluster = row["count"]
            
            dominant_row = dominant_mapping[dominant_mapping["cluster_id"] == cluster_id].iloc[0]
            dominant_class = dominant_row["name_epist"]
            dominance = dominant_row["dominance"]
            total_in_cluster = dominant_row["total"]
            
            missing_suggestions.append({
                "ID_epistem": epistem_id,
                "name_epist": epistem_name,
                "cluster_id": cluster_id,
                "count_in_cluster": count_in_cluster,
                "new_dominance": (count_in_cluster / total_in_cluster),
                "dominant_class": dominant_class,
                "original_dominance": dominance
            })
    
    suggestions_df = pd.DataFrame(missing_suggestions)
    
    # Print grouped by missing class
    for epistem_id in sorted(missing_epistem):
        subset = suggestions_df[suggestions_df["ID_epistem"] == epistem_id]
        print(f"\n {subset['name_epist'].iloc[0]} (ID: {epistem_id})")
        print(subset[[
            "cluster_id", 
            "count_in_cluster", 
            "new_dominance", 
            "dominant_class", 
            "original_dominance"
        ]].to_string(index=False))
        
        # Suggestion
        best_cluster = subset.iloc[0]
        print(f" Suggest assignment to cluster {best_cluster['cluster_id']} "
              f"({best_cluster['count_in_cluster']} instances)")

high = (dominant_mapping["dominance"] >= 0.8).sum()
medium = ((dominant_mapping["dominance"] >= 0.6) & (dominant_mapping["dominance"] < 0.8)).sum()
low = (dominant_mapping["dominance"] < 0.6).sum()

print("\n===== LABEL QUALITY DISTRIBUTION =====")
print(f"High confidence clusters: {high}")
print(f"Medium confidence clusters: {medium}")
print(f"Low/Unmapped clusters: {low}")


===== OVERALL SUMMARY =====
Mean cluster dominance: 0.476
Min dominance: 0.263
Max dominance: 1.000

===== LABEL RECOMMENDATION =====
 cluster_id               name_epist  dominant_count  total  dominance  confidence_label
          0             Cleared Land               7     18   0.388889    NO CLEAR CLASS
          1 Secondary Dryland Forest              20     38   0.526316    LOW_CONFIDENCE
          2               Settlement              50    168   0.297619    NO CLEAR CLASS
          3     Oil palm monoculture              20     43   0.465116    LOW_CONFIDENCE
          4               Settlement              11     40   0.275000    NO CLEAR CLASS
          5   Secondary Swamp Forest              10     33   0.303030    NO CLEAR CLASS
          6     Oil palm monoculture              28     32   0.875000   HIGH_CONFIDENCE
          7   Secondary Swamp Forest               2      2   1.000000   HIGH_CONFIDENCE
          8   Secondary Swamp Forest              22     57   0.